# V00_export: migration flows, nodes and per-person paths (layer 4)
Task `tasks/T-005-v00-export-migration.md` (brief B-016). SQL + pandas only, no ML/LLM. Reads `data/processed/`, writes only `web/data/`.

| Step | Purpose |
|---|---|
| 0.1 | Stops: `residences` + `households.served_state` as a `served` stop, ordered per NAID × person |
| 0.2 | Legs: consecutive stops where the place changes; Act cohort from `application_year` |
| 0.3 | `flows` (state → state, county rows kept) and `nodes` (people left / arrived) |
| 0.4 | Counts: legs per kind pair, people per cohort, files with ≥2 stops |
| 0.5 | Assertions, write `web/data/flows.json`, `nodes.json`, `paths.json` |

**Sources:** `data/processed/residences.parquet` (111,317 rows), `households.parquet` (78,745 files), `places.parquet` (1,919 places; A05–A08, 2026-11-22).

**Rules (T-005, PLAN §3):** all claim types count (S, W, R, BLWT); each leg counts once; no decade bins; Pension Acts only as an application cohort.

**Caveats** (`docs/DATA.md`): residences come from regex over declarations (A07: 83% row-weighted strict). Most stops are undated; interim stops (`removed_to`, `lived_since`, `married`) are ordered by date, then page, so their order is a best guess when undated. In W files the widow's chain is her own (`person = widow`), not the veteran's. `served` is the state of the line he served in, not always where he lived. MA → ME is partly administrative (Maine ⊂ MA until 1820).

In [1]:
import sys, json
from pathlib import Path
PIPELINE_DIR = Path.cwd().parent
if str(PIPELINE_DIR) not in sys.path: sys.path.insert(0, str(PIPELINE_DIR))
import config
import duckdb, pandas as pd
pd.set_option('display.width', 220); pd.set_option('display.max_rows', 120)
con = duckdb.connect()
for t in ["residences", "households", "places"]:
    con.sql(f"create view {t} as select * from read_parquet('{config.PROCESSED_DIR / (t + '.parquet')}')")
con.sql(open(config.PIPELINE_DIR / "schema.sql").read())
def q(sql): return con.sql(sql).df()
WEB_DATA = config.PROJECT_DIR / "web" / "data"
# DuckDB 2.0: `//` on DOUBLE is float division. Dates and pages are whole numbers stored as DOUBLE: check, then cast to bigint.
assert q("select count(*) n from residences where date <> floor(date) or pageNumber <> floor(pageNumber)").n[0] == 0
q("select (select count(*) from residences) residences, (select count(*) from households) households, (select count(*) from places) places")

,residences,households,places
0,111317,78745,1919


## 0.1 Stops
One row per NAID × person × stop. Residence rows keep their A05 order (`born` → `enlisted` → interim → `now_resides`); the veteran's `served_state` (not `Cont`, not null) goes in as a `served` stop right after `enlisted`, with no date or page. Only S, W, R and BLWT files; the other file types (roll headers, blank numbers, Not Rev, Old War …) are dropped. The 9,350 `most_frequent` fallback rows stay in, as `person = unknown`, `method = most_frequent`.

In [2]:
CLAIMS = "('S', 'W', 'R', 'BLWT')"
RANK = "case kind when 'born' then 0 when 'enlisted' then 1 when 'served' then 2 when 'now_resides' then 4 else 3 end"
con.sql(f'''
create or replace table stops as
with s as (
  select r.NAID, r.person, r.seq as src_seq, r.place_id, r.state, r.precision, r.kind,
         r.date::bigint as date, r.pageNumber::bigint as page, r.method
  from residences r join households h using (NAID) where h.claim_type in {CLAIMS}
  union all
  select NAID, 'veteran', null, served_state, served_state, 'state', 'served', null, null, 'served_state'
  from households where claim_type in {CLAIMS} and served_state is not null and served_state <> 'Cont')
select NAID, person, row_number() over (partition by NAID, person order by {RANK}, src_seq) as seq,
       place_id, state, precision, kind, date, page, method
from s''')
q("select h.claim_type, count(*) residence_rows, count(distinct NAID) files from residences r join households h using (NAID) group by 1 order by 2 desc")

,claim_type,residence_rows,files
0,S,49654,32669
1,W,43909,20871
2,R,15846,9102
3,BLWT,1540,1399
4,number_blank,185,121
5,other,71,61
6,not_rev,60,48
7,old_war,27,24
8,roll_header,13,8
9,single_document,12,12


In [3]:
q("select kind, count(*) stops, count(date) dated, count(*) filter (where precision = 'county') county_level from stops group by 1 order by 2 desc")

,kind,stops,dated,county_level
0,now_resides,74622,57791,31252
1,served,70853,0,0
2,born,14208,10847,6455
3,removed_to,10909,2435,4703
4,married,6170,2728,2468
5,enlisted,4768,832,2057
6,lived_since,272,29,84


## 0.2 Legs
A leg is two consecutive stops of one person where the place changes: compared by county when both ends are counties, else by state. So `cts_hartford → CT` is not a leg, and `nys_tioga → nys_albany` is (a within-state leg). `move_after` / `move_before` are the two stops' dates (a range, often open).
Act cohort from `households.application_year`: 1818–19, 1820–31, 1832–35, 1836–47, 1848+; `unknown` when the file has no application year.

In [4]:
COHORT = '''case when application_year is null then 'unknown'
  when application_year <= 1819 then '1818-19' when application_year <= 1831 then '1820-31'
  when application_year <= 1835 then '1832-35' when application_year <= 1847 then '1836-47' else '1848+' end'''
con.sql(f'''
create or replace table people as
select distinct s.NAID, s.person, h.claim_type, h.veteran_name, h.application_year, {COHORT} as act_cohort
from stops s join households h using (NAID)''')
con.sql('''
create or replace table legs as
with s as (
  select *, lag(seq) over w as from_seq, lag(place_id) over w as from_place_id, lag(state) over w as from_state,
         lag(precision) over w as from_precision, lag(kind) over w as from_kind, lag(date) over w as from_date
  from stops window w as (partition by NAID, person order by seq))
select s.NAID, s.person, from_seq, seq as to_seq, from_place_id, place_id as to_place_id, from_state, state as to_state,
       (from_precision = 'county' and precision = 'county') as county_level, from_kind, kind as to_kind,
       from_date as move_after, date as move_before, p.claim_type, p.act_cohort
from s join people p using (NAID, person)
where from_seq is not null
  and case when from_precision = 'county' and precision = 'county' then from_place_id <> place_id else from_state <> state end''')
q('''select count(*) legs, count(*) filter (where county_level) county_level, count(*) filter (where from_state = to_state) within_state,
       count(*) filter (where from_kind = 'served' or to_kind = 'served') touch_served,
       count(distinct (NAID, person)) people, count(distinct NAID) files from legs''')

,legs,county_level,within_state,touch_served,people,files
0,39229,3324,1485,27547,30642,28945


**Check against the `moves` view** (`schema.sql`): with the `served` stops left out and only state changes counted, the legs must equal the `moves` rows of S, W, R and BLWT files.

In [5]:
chk = q(f'''
with s as (select * from stops where kind <> 'served'),
l as (select state, lag(state) over (partition by NAID, person order by seq) as fs from s)
select (select count(*) from l where fs is not null and fs <> state) as legs_no_served,
       (select count(*) from moves m join households h using (NAID) where h.claim_type in {CLAIMS}) as moves_view''')
assert chk.legs_no_served[0] == chk.moves_view[0], chk
chk

,legs_no_served,moves_view
0,18868,18868


## 0.3 Flows and nodes
**`flows`**: one row per from × to × Act cohort. `level = state` rows cover every leg (within-state legs show as `from = to`); `level = county` rows repeat the legs where both ends are counties, for the click-a-state drill-down. `legs` = leg count, `people` = distinct NAID × person, `served_legs` = legs that start or end at a `served` stop (drop them to see residences only), `claim_types` = legs per claim type.

**`nodes`**: one row per place × Act cohort. State nodes count people with a leg out of / into the state from / to another state (within-state legs excluded). County nodes count people with a leg out of / into the county (any leg with a county at that end). lat/lon from `places`. Cohorts partition people (one application year per file), so counts add up across cohorts.

In [6]:
def flows_at(level, a, b, where):
    return q(f'''
    with f as (select {a} as "from", {b} as "to", from_state, to_state, act_cohort, claim_type, NAID, person,
                      (from_kind = 'served' or to_kind = 'served') as sv from legs where {where})
    select '{level}' as level, "from", "to", from_state, to_state, act_cohort, count(*) as legs,
           count(distinct (NAID, person)) as people, count(*) filter (where sv) as served_legs
    from f group by all''')
ct = lambda a, b, where: q(f'''select {a} as "from", {b} as "to", act_cohort, claim_type, count(*) n
                               from legs where {where} group by all''')
flows = []
for level, a, b, where in [("state", "from_state", "to_state", "true"), ("county", "from_place_id", "to_place_id", "county_level")]:
    f = flows_at(level, a, b, where)
    c = (ct(a, b, where).pivot_table(index=["from", "to", "act_cohort"], columns="claim_type", values="n", fill_value=0)
           .apply(lambda r: {k: int(v) for k, v in r.items() if v}, axis=1).rename("claim_types").reset_index())
    flows.append(f.merge(c, on=["from", "to", "act_cohort"], how="left"))
flows = pd.concat(flows, ignore_index=True)
flows.groupby("level").agg(rows=("legs", "size"), legs=("legs", "sum"), people=("people", "sum"))

,rows,legs,people
level,,,
county,3114,3324,3320
state,1778,39229,38683


In [7]:
flows[flows.level == "state"].groupby(["from", "to"]).legs.sum().sort_values(ascending=False).head(12)

from  to
CT    NY    2991
MA    ME    2231
      NY    2012
VA    KY    1581
MA    VT    1522
      NH    1024
CT    VT     977
VA    NC     892
NC    TN     892
VA    TN     787
CT    MA     719
PA    OH     658
Name: legs, dtype: int64

In [8]:
nodes = q('''
with e as (
  select 'state' as level, from_state as place_id, act_cohort, NAID, person, 'left' as dir from legs where from_state <> to_state
  union all select 'state', to_state, act_cohort, NAID, person, 'arrived' from legs where from_state <> to_state
  union all select 'county', from_place_id, act_cohort, NAID, person, 'left' from legs l join places p on p.place_id = l.from_place_id where p.county is not null
  union all select 'county', to_place_id, act_cohort, NAID, person, 'arrived' from legs l join places p on p.place_id = l.to_place_id where p.county is not null)
select e.level, e.place_id, p.state, p.county, p.lat, p.lon, e.act_cohort,
       count(distinct (NAID, person)) filter (where dir = 'left') as people_left,
       count(distinct (NAID, person)) filter (where dir = 'arrived') as people_arrived
from e join places p using (place_id) group by all order by level, place_id, act_cohort''')
nodes.groupby("level").agg(rows=("place_id", "size"), places=("place_id", "nunique"), left=("people_left", "sum"), arrived=("people_arrived", "sum"))

,rows,places,left,arrived
level,,,,
county,3166,1086,6329,16467
state,177,33,36838,36886


## 0.4 Counts
Legs per kind pair (from kind → to kind), all levels.

In [9]:
q("select from_kind, to_kind, count(*) legs from legs group by all order by legs desc").pivot_table(
    index="from_kind", columns="to_kind", values="legs", fill_value=0, margins=True, aggfunc="sum")

to_kind,born,enlisted,lived_since,married,now_resides,removed_to,served,All
from_kind,,,,,,,,
born,157,1241,2,96,349,151,3008,5004
enlisted,0,101,0,45,256,62,958,1422
lived_since,0,0,2,6,55,5,0,68
married,0,0,7,148,3041,296,0,3492
removed_to,0,0,25,257,3755,1625,0,5662
served,0,0,79,294,17696,5512,0,23581
All,157,1342,115,846,25152,7651,3966,39229


People per Act cohort: all people with stops, people with ≥2 stops, people with ≥1 leg.

In [10]:
q('''
with n as (select NAID, person, count(*) stops from stops group by all),
l as (select distinct NAID, person from legs)
select p.act_cohort, count(*) people, count(*) filter (where n.stops >= 2) people_2plus_stops,
       count(l.NAID) people_with_legs, count(distinct p.NAID) files
from people p join n using (NAID, person) left join l using (NAID, person) group by 1 order by 1''')

,act_cohort,people,people_2plus_stops,people_with_legs,files
0,1818-19,2586,1311,633,2073
1,1820-31,11895,6990,4257,9962
2,1832-35,24445,19735,11959,21780
3,1836-47,26740,10383,6626,15082
4,1848+,22563,10475,6696,12148
5,unknown,16095,833,471,13986


Files with ≥2 stops: counted per file (all people together) and per person (a chain of their own).

In [11]:
q('''
with f as (select NAID, sum(n) stops, max(n) max_person from
             (select NAID, person, count(*) n from stops group by all) group by 1)
select count(*) files_with_stops, count(*) filter (where stops >= 2) files_2plus_stops,
       count(*) filter (where max_person >= 2) files_person_2plus_stops,
       (select count(distinct NAID) from legs) files_with_legs from f''')

,files_with_stops,files_2plus_stops,files_person_2plus_stops,files_with_legs
0,75031,61377,46683,28945


## 0.5 Assertions and export
`paths.json`: one record per NAID × person with its ordered stops (`seq`, `place_id`, `state`, `precision`, `kind`, `date`, `page`; `method` only on the fallback and `served` stops). Split by state only if the file passes 20 MB: then each person goes to `paths/<state>.json` by the state of their last stop, and `paths.json` is the index of parts.

In [12]:
pl_ids = set(q("select place_id from places").place_id)
n_legs, n_cty = q("select count(*) a, count(*) filter (where county_level) b from legs").iloc[0]
# unique keys, no null keys
assert q("select count(*) n from (select NAID, person, seq from stops group by all having count(*) > 1)").n[0] == 0
assert q("select count(*) n from stops where NAID is null or person is null or seq is null or place_id is null or kind is null").n[0] == 0
assert q("select count(*) n from (select NAID, person, from_seq from legs group by all having count(*) > 1)").n[0] == 0
assert q("select count(*) n from (select NAID, person from people group by all having count(*) > 1)").n[0] == 0
assert not flows.duplicated(["level", "from", "to", "act_cohort"]).any()
assert not nodes.duplicated(["level", "place_id", "act_cohort"]).any()
# every place_id is in places
assert set(q("select distinct place_id from stops").place_id) <= pl_ids
assert set(flows["from"]) | set(flows["to"]) <= pl_ids and set(nodes.place_id) <= pl_ids
assert nodes[["lat", "lon"]].notna().all().all()
# leg totals = sum of flows
assert flows[flows.level == "state"].legs.sum() == n_legs
assert flows[flows.level == "county"].legs.sum() == n_cty
assert (flows.claim_types.map(lambda d: sum(d.values())) == flows.legs).all()
assert (flows.people <= flows.legs).all() and (flows.served_legs <= flows.legs).all()
# only the four claim types, cohorts as defined
assert set(q("select distinct claim_type from people").claim_type) <= {"S", "W", "R", "BLWT"}
assert set(flows.act_cohort) <= {"1818-19", "1820-31", "1832-35", "1836-47", "1848+", "unknown"}
print("ok:", int(n_legs), "legs ·", int(n_cty), "county-level ·", len(flows), "flow rows ·", len(nodes), "node rows")

ok: 39229 legs · 3324 county-level · 4892 flow rows · 3343 node rows


In [13]:
def clean(v):
    if isinstance(v, float) and v != v: return None
    if hasattr(v, "item"): return v.item()
    return v
def records(df): return [{k: clean(v) for k, v in r.items()} for r in df.to_dict("records")]

st = q("select * from stops order by NAID, person, seq")
st["method"] = st.method.where(st.method.isin(["most_frequent", "served_state"]))
STOP_COLS = ["seq", "place_id", "state", "precision", "kind", "date", "page", "method"]
by_person = {k: [{c: v for c, v in s.items() if v is not None} for s in records(g[STOP_COLS])]
             for k, g in st.groupby(["NAID", "person"], sort=False)}
ppl = q("select * from people order by NAID, person")
paths = [{"NAID": r["NAID"], "person": r["person"], "name": r["veteran_name"], "claim_type": r["claim_type"],
          "application_year": r["application_year"], "act_cohort": r["act_cohort"], "stops": by_person[(r["NAID"], r["person"])]}
         for r in records(ppl)]

WEB_DATA.mkdir(parents=True, exist_ok=True)
def dump(obj, name):
    p = WEB_DATA / name
    p.write_text(json.dumps(obj, separators=(",", ":"), allow_nan=False))
    return p, round(p.stat().st_size / 1e6, 2)
out = [dump(records(flows), "flows.json"), dump(records(nodes), "nodes.json")]
size = len(json.dumps(paths, separators=(",", ":"), allow_nan=False).encode()) / 1e6
print("paths as one file:", round(size, 1), "MB")
if size <= 20:
    out.append(dump(paths, "paths.json"))
else:
    # split by the state of each person's last stop (where the file was filed from); paths.json becomes the index
    (WEB_DATA / "paths").mkdir(exist_ok=True)
    parts = {}
    for r in paths: parts.setdefault(r["stops"][-1]["state"], []).append(r)
    index = []
    for stt, rs in sorted(parts.items()):
        fn = "paths/" + stt.replace(" ", "_") + ".json"
        _, mb = dump(rs, fn)
        index.append({"state": stt, "file": fn, "people": len(rs), "stops": sum(len(r["stops"]) for r in rs), "mb": mb})
    out.append(dump({"split_by": "state of last stop", "parts": index}, "paths.json"))
    print(pd.DataFrame(index).sort_values("mb", ascending=False).head(8).to_string(index=False))
out

paths as one file: 32.5 MB


state          file  people  stops   mb
   NY paths/NY.json   17593  33275 5.74
   MA paths/MA.json   14556  20766 4.07
   VA paths/VA.json    9011  14117 2.65
   CT paths/CT.json    9225  13579 2.63
   PA paths/PA.json    7027  11495 2.12
   VT paths/VT.json    4803   9980 1.65
   NH paths/NH.json    5430   8480 1.60
   NC paths/NC.json    5317   8606 1.59


[(PosixPath('/Users/larawashington/Documents/Parsons/MajorStudio1/Projects/revolutionary-xroads/web/data/flows.json'),
  0.83),
 (PosixPath('/Users/larawashington/Documents/Parsons/MajorStudio1/Projects/revolutionary-xroads/web/data/nodes.json'),
  0.55),
 (PosixPath('/Users/larawashington/Documents/Parsons/MajorStudio1/Projects/revolutionary-xroads/web/data/paths.json'),
  0.0)]

In [14]:
# round trip: the files on disk hold what the tables hold
f = json.loads((WEB_DATA / "flows.json").read_text()); n = json.loads((WEB_DATA / "nodes.json").read_text())
p = json.loads((WEB_DATA / "paths.json").read_text())
if "parts" in p:
    p = [r for part in p["parts"] for r in json.loads((WEB_DATA / part["file"]).read_text())]
assert len(p) == len(ppl) and sum(len(r["stops"]) for r in p) == len(st)
assert len({(r["NAID"], r["person"]) for r in p}) == len(p)
assert all([s["seq"] for s in r["stops"]] == list(range(1, len(r["stops"]) + 1)) for r in p)
print(len(f), "flows ·", len(n), "nodes ·", len(p), "people ·", len(st), "stops")

4892 flows · 3343 nodes · 104324 people · 181802 stops


## Counts (run 2026-11-23)
- Stops: 181,802 for 104,324 people in 75,031 files (S, W, R, BLWT); 70,853 are `served` stops. 61,377 files have ≥2 stops; 46,683 have one person with ≥2 stops.
- Legs: **39,229** for 30,642 people in 28,945 files; 3,324 county-level; 1,485 within one state; 27,547 (70%) start or end at a `served` stop. Without `served`: 18,868 legs = the `moves` view for these claim types.
- Largest kind pairs: served → now_resides 17,696 · served → removed_to 5,512 · removed_to → now_resides 3,755 · married → now_resides 3,041 · born → served 3,008.
- People with legs by cohort: 1818–19 633 · 1820–31 4,257 · 1832–35 11,959 · 1836–47 6,626 · 1848+ 6,696 · unknown 471.
- Files: `flows.json` 4,892 rows (1,778 state, 3,114 county), `nodes.json` 3,343 rows (177 state, 3,166 county), paths 32.5 MB as one file, so split into 33 files in `paths/` by the state of the last stop (largest NY 5.7 MB); `paths.json` is the index.